# COMPANY U Book Frequency AnalysisExtract books from user checkout data, clean titles, count frequency, and generate book distribution report.

In [1]:
import pandas as pdimport refrom collections import defaultdictimport warningswarnings.filterwarnings('ignore')print(" Libraries imported successfully")

 Libraries imported successfully

## Title Cleaning FunctionDefine function to clean book titles:- Strip whitespace (leading/trailing)- Normalize unicode (NFKC)- Remove special characters/symbols- Collapse multiple spaces- Convert to lowercase

In [2]:
def clean_title(raw_title):    """    Clean book title:    1. Strip leading/trailing whitespace    2. Normalize unicode (NFKC)    3. Remove all special characters/symbols (keep only alphanumeric and spaces)    4. Collapse multiple spaces into one    5. Convert to lowercase    """    # Step 1: Strip leading/trailing whitespace    title = raw_title.strip()        # Step 2: Normalize unicode    title = pd.Series(title).str.normalize('NFKC')[0]        # Step 3: Remove special characters - keep only alphanumeric and spaces    title = re.sub(r'[^a-zA-Z0-9\s]', '', title)        # Step 4: Collapse multiple spaces into single space    title = re.sub(r'\s+', ' ', title).strip()        # Step 5: Convert to lowercase    title = title.lower()        return titleprint(" clean_title() function defined")

 clean_title() function defined

## Load COMPANY U User Data

In [3]:
# Read the COMPANY U CSV filedf = pd.read_csv('company_u.csv')print("=== COMPANY U DATASET ===\n")print(f"Total users: {len(df)}")print(f"\nFirst 5 users:")print(df.head())

=== COMPANY U DATASET ===Total users: 298First 5 users:        id                                              books0  User002  A guide to the project management body of know...1  User003       Ideology : an introduction / Terry Eagleton.2  User004  How to think on your feet : a revolutionary te...3  User006  Homicide / by Martin Daly and Margo Wilson., E...4  User007  Far from home : reading and word study / by Wi...

## Process Books and Count FrequencyExtract books from each user's library, clean titles, and count checkouts.

In [4]:
# Dictionary to count book occurrencesbook_counts = defaultdict(int)# Process each user's booksfor idx, row in df.iterrows():    books_str = row['books']        # Skip if no books    if pd.isna(books_str) or not books_str.strip():        continue        # Split by "., " to separate individual books    books_list = books_str.split('., ')        for book_entry in books_list:        # Clean up the entry        book_entry = book_entry.strip()        if not book_entry:            continue                # Remove trailing period or comma        if book_entry.endswith('.'):            book_entry = book_entry[:-1]        elif book_entry.endswith(','):            book_entry = book_entry[:-1]                # Extract book title (before the "/" character)        if '/' in book_entry:            book_title = book_entry.split('/')[0].strip()        else:            book_title = book_entry.strip()                # CLEAN TITLE before counting        book_title = clean_title(book_title)                # Skip empty titles after cleaning        if not book_title:            continue                # Count the book        book_counts[book_title] += 1print(f" Processed all users and extracted books")

 Processed all users and extracted books

## Create Book Frequency DataFrame

In [5]:
# Convert to DataFrame and sort by countbooks_df = pd.DataFrame(list(book_counts.items()), columns=['Title', 'Number'])books_df = books_df.sort_values('Number', ascending=False).reset_index(drop=True)print(f"Total unique books: {len(books_df)}")print(f"Total book checkouts: {books_df['Number'].sum()}")print("\nTop 20 most checked out books:")print(books_df.head(20))

Total unique books: 666Total book checkouts: 718Top 20 most checked out books:                                                Title  Number0                             the bastard of istanbul       31   business model generation a handbook for visio...       32                                     college algebra       33                                  kafka on the shore       34         the norton anthology of american literature       35                                   international law       26   the first republic of armenia 19181920 on its ...       27                       one hundred years of solitude       28          marketing management and strategy a reader       29                                              lolita       210                                        the tempest       211                              the craft of research       212  cryptography and network security principles a...       213  introduction to emergency management and disas...       214     

## Save Results to CSV

In [6]:
# Add summary rowsummary_row = pd.DataFrame({'Title': ['SUMMARY'], 'Number': [books_df['Number'].sum()]})books_df_with_summary = pd.concat([books_df, summary_row], ignore_index=True)# Save to CSV filebooks_df_with_summary.to_csv('company_u_books_output.csv', index=False, quoting=1)print(" Created file: company_u_books_output.csv")print(f" Total rows: {len(books_df_with_summary)} (including summary)")print("\nSummary row:")print(summary_row)

 Created file: company_u_books_output.csv Total rows: 667 (including summary)Summary row:     Title  Number0  SUMMARY     718

## Summary Statistics

In [7]:
print("\n" + "="*60)print("COMPANY U BOOK FREQUENCY ANALYSIS SUMMARY")print("="*60)print(f"Total users analyzed: {len(df)}")print(f"Total unique books: {len(books_df)}")print(f"Total book checkouts: {int(books_df['Number'].sum())}")print(f"Average checkouts per book: {books_df['Number'].mean():.2f}")print(f"Most checked out book: '{books_df.iloc[0]['Title']}' ({int(books_df.iloc[0]['Number'])} checkouts)")print("="*60)

============================================================COMPANY U BOOK FREQUENCY ANALYSIS SUMMARY============================================================Total users analyzed: 298Total unique books: 666Total book checkouts: 718Average checkouts per book: 1.08Most checked out book: 'the bastard of istanbul' (3 checkouts)============================================================

## Top 5 Most Checked Out Books

In [8]:
top_5_books = books_df.head(5).reset_index(drop=True)top_5_books.index = top_5_books.index + 1print("\n TOP 5 MOST CHECKED OUT BOOKS IN COMPANY U LIBRARY\n")print(top_5_books.to_string())print("\n")for idx, row in top_5_books.iterrows():    print(f"{idx}. '{row['Title']}' — {int(row['Number'])} checkouts")

 TOP 5 MOST CHECKED OUT BOOKS IN COMPANY U LIBRARY                                                                                Title  Number1                                                             the bastard of istanbul       32  business model generation a handbook for visionaries game changers and challengers       33                                                                     college algebra       34                                                                  kafka on the shore       35                                         the norton anthology of american literature       31. 'the bastard of istanbul' — 3 checkouts2. 'business model generation a handbook for visionaries game changers and challengers' — 3 checkouts3. 'college algebra' — 3 checkouts4. 'kafka on the shore' — 3 checkouts5. 'the norton anthology of american literature' — 3 checkouts